# JDV Batch Runner — Google Colab

Ejecuta el **Vagueness Judge entrenado** (Qwen2.5-3B + LoRA) sobre las 150 queries de evaluación **sin** correr el pipeline CAO completo.

## Requisitos previos

1. GPU en Colab (T4 o mejor)
2. Subir el adapter LoRA a Google Drive (`Qwen2.5-3B-Instruct-Vagueness_Judge/`)
3. Opcional: `HF_TOKEN` si HuggingFace lo requiere

## Salida

JSONs en `experiments/cao/data/jdv_results/` — descárgalos y en local:

```bash
uv run python experiments/cao/batch_cao.py --jdv-dir experiments/cao/data/jdv_results
```

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Configuración (editar) ---
REPO_URL = "https://github.com/TU_USUARIO/Tesis.git"  # o clonar manualmente
REPO_DIR = "/content/Tesis"

ADAPTER_PATH = "/content/drive/MyDrive/Tesis/adapters/Qwen2.5-3B-Instruct-Vagueness_Judge"
# Alternativa: modelo mergeado completo (~6 GB)
MERGED_MODEL_PATH = None  # ej: "/content/drive/MyDrive/Tesis/merged/Qwen2.5-3B-Instruct-Vagueness_Judge"

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
INPUT_JSON = f"{REPO_DIR}/experiments/cao/data/evaluation_sample_v1.json"
OUTPUT_DIR = f"{REPO_DIR}/experiments/cao/data/jdv_results"
LIMIT = None  # ej: 1 para smoke test, None para las 150
SKIP_EXISTING = True

In [ ]:
import os
from pathlib import Path

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print("Working dir:", os.getcwd())

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from experiments.cao.jdv_runner import run_batch
from pathlib import Path

manifest = run_batch(
    input_path=Path(INPUT_JSON),
    output_dir=Path(OUTPUT_DIR),
    base_model=BASE_MODEL,
    adapter_path=None if MERGED_MODEL_PATH else ADAPTER_PATH,
    merged_model_path=MERGED_MODEL_PATH,
    limit=LIMIT,
    skip_existing=SKIP_EXISTING,
)
print(f"Manifest entries: {len(manifest)}")

In [ ]:
# Verificar un resultado
import json
from pathlib import Path

files = sorted(Path(OUTPUT_DIR).glob("jdv_*.json"))
print(f"Total files: {len(files)}")
if files:
    sample = json.loads(files[0].read_text(encoding="utf-8"))
    print(json.dumps(sample, indent=2, ensure_ascii=False)[:1500])

In [ ]:
# Empaquetar resultados para descargar
import shutil
from pathlib import Path

zip_path = "/content/jdv_results.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", OUTPUT_DIR)
print(f"Created {zip_path}")

from google.colab import files
files.download(zip_path)

---
## Modo API Server

Alternativa: servir el modelo JDV como API para que `jdv_eval_v2.py` se conecte desde tu máquina local y maneje las rondas de clarificación interactivamente.

**Requisito:** Cuenta gratuita en [ngrok.com](https://ngrok.com) y un token de autenticación (`NGROK_AUTH_TOKEN`).

In [ ]:
# --- Instalar dependencias para el servidor ---
!pip install -q flask flask-cors pyngrok

In [ ]:
import os

# Configura tu token de ngrok aquí (o usa la variable de entorno en Colab secrets)
NGROK_AUTH_TOKEN = ""  # <-- PON TU TOKEN AQUÍ
if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok autenticado")

In [ ]:
# --- Cargar el modelo JDV (desde el adapter en Drive) ---
import sys
sys.path.insert(0, REPO_DIR)

from experiments.cao.jdv_runner import load_jdv_model

print("Cargando modelo JDV...")
model, tokenizer = load_jdv_model(
    base_model=BASE_MODEL,
    adapter_path=ADAPTER_PATH,
    merged_model_path=MERGED_MODEL_PATH,
)
print("Modelo listo")

In [ ]:
# --- Iniciar servidor Flask en background ---
import json
import threading
from flask import Flask, request, jsonify

app = Flask(__name__)


@app.route("/v1/chat/completions", methods=["POST"])
def chat_completions():
    data = request.get_json()
    messages = data.get("messages", [])

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    import torch
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=512,
            temperature=0.2,
            top_p=0.95,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id or tokenizer.pad_token_id,
        )

    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if full.startswith(prompt):
        content = full[len(prompt):].strip()
    else:
        content = full.strip()

    return jsonify({"choices": [{"message": {"content": content}}]})


def run_server():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)


thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("Servidor JDV iniciado en http://localhost:5000")

In [ ]:
# --- Exponer con ngrok ---
if NGROK_AUTH_TOKEN:
    public_url = ngrok.connect(5000, domain="free")
    print(f"\n{'='*60}")
    print(f"JDV API URL (para jdv_eval_v2.py):")
    print(f"  export VAGUE_ENDPOINT_URL={public_url}")
    print(f"{'='*60}\n")
else:
    print("[WARN] NGROK_AUTH_TOKEN no configurado — el servidor solo es accesible localmente en Colab")
    print("Configúralo en la celda anterior y vuelve a ejecutar.")